# Fraud & Anomaly Detection — EDA & Modeling

Dataset: Kaggle Credit Card Fraud Detection (ULB) — real anonymized European transactions, PCA-transformed features V1–V28, Time, Amount, Class.

Sampled for repo size: all 492 real frauds retained + 40,000 sampled legitimate transactions (40,492 rows total, ~1.2% fraud rate in this sample vs ~0.17% in the full 284,807-row original).

Goal: Detect fraudulent transactions WITHOUT using labels during training (unsupervised anomaly detection) — labels are used only for evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

df = pd.read_csv("../data/credit_card_transactions.csv")
df.shape

In [ ]:
print(df["Class"].value_counts())
print(f"Fraud rate: {df['Class'].mean()*100:.3f}%")

sns.countplot(data=df, x="Class", palette=["#4facfe", "#ff6b8b"])
plt.title("Class Balance (0 = Normal, 1 = Fraud)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Class", y="Amount", ax=axes[0], palette=["#4facfe", "#ff6b8b"])
axes[0].set_title("Amount by Class (with outliers)")

sns.boxplot(data=df[df["Amount"] < 500], x="Class", y="Amount", ax=axes[1], palette=["#4facfe", "#ff6b8b"])
axes[1].set_title("Amount by Class (Amount < 500, zoomed in)")
plt.tight_layout()
plt.show()

In [ ]:
df["Hour"] = (df["Time"] // 3600) % 24
hourly_fraud = df.groupby("Hour")["Class"].mean() * 100

plt.figure(figsize=(10, 4))
hourly_fraud.plot(kind="line", marker="o", color="#ff6b8b")
plt.title("Fraud Rate by Hour of Day")
plt.ylabel("Fraud Rate (%)")
plt.show()

In [ ]:
correlations = df.corr(numeric_only=True)["Class"].drop("Class").sort_values()
top_corr = pd.concat([correlations.head(5), correlations.tail(5)])

plt.figure(figsize=(8, 5))
top_corr.plot(kind="barh", color=["#4facfe" if v < 0 else "#ff6b8b" for v in top_corr])
plt.title("Top 10 Features Most Correlated with Fraud (V1-V28)")
plt.show()

## Key EDA Findings

- Extreme class imbalance even after sampling (~1.2% fraud) — genuine real-world fraud detection challenge
- Fraudulent transactions show a different Amount distribution, though with meaningful overlap with legitimate ones — Amount alone isn't a reliable fraud signal
- Fraud rate fluctuates by hour, suggesting some time-of-day pattern worth feature engineering in future iterations
- Several PCA components (V14, V17, V12, V10, V4 in this dataset) show much stronger correlation with fraud than others — informs which features matter most, even though they're anonymized

## Modeling Summary

Trained an **Isolation Forest** (unsupervised) on all V1-V28 + Amount features, WITHOUT using the Class label. Isolation Forest works by isolating anomalies — points that are "few and different" — with fewer random partitions than normal points, making it well-suited to fraud detection where fraud patterns are rare and often unlabeled in the real world.

Fraud labels were used only afterward to evaluate how well the model's anomaly flags aligned with genuine fraud cases (see `src/anomaly_model.py` for the training + evaluation code and printed classification report / ROC-AUC).

A `SuspiciousTransactionError` custom exception was also implemented (`src/exceptions.py`) to demonstrate real-time critical-anomaly flagging with proper exception handling, in addition to the batch-scoring dashboard.